# Install the dependencies using bundler

In [13]:
require 'bundler/inline'

gemfile do
  source 'https://rubygems.org'
    
  gem 'daru', '~> 0.3'
  gem 'daru-view', '~> 0.2.6'
  gem "nyaplot"
  gem 'matrix', '~> 0.4.2'
  gem 'geocoder'
  gem 'gpx'
end
require 'daru'
require 'daru/view'

false

# Analize a foil session

In [4]:
require_relative 'foil_activity_analyzer'

file = 'activities/2024-06-17_2.gpx'
lift_speed = 12
fa = FoilActivityAnalyzer.new(file, lift_speed)
df = fa.data_frame
df.head(10)
df[:distance_m].replace_values(0, 0.000000001)
df[:distance_cumsum_m] = df[:distance_m].cumsum
df.head 5

,timestamp,lat,lon,hr,ellapsed_sec,distance_m,speed_m_s,distance_cumsum_m,on_foil
0,2024-06-15 11:22:15 UTC,52.46734491549432,5.33144898712635,97,0.0,1.0e-09,0.0,1.0e-09,0
1,2024-06-15 11:22:17 UTC,52.46731549501419,5.3314343187958,98,2.0,3.4189852354446244,1.71,3.4189852364446245,0
2,2024-06-15 11:22:20 UTC,52.46731985360384,5.331416800618172,98,3.0,1.2818592877714723,0.43,4.700844524216096,0
3,2024-06-15 11:22:22 UTC,52.46731708757579,5.331427529454231,99,2.0,0.7891882503677813,0.39,5.490032774583877,0
4,2024-06-15 11:22:23 UTC,52.46731205843389,5.331429373472929,98,1.0,0.5729970700844821,0.57,6.0630298446683595,0


Lehet az adatokat ido es tavolsag alapjan indexelni, de nem igazan tudom hogy mire jo az index

[Basic time seriies index](https://nbviewer.org/github/SciRuby/sciruby-notebooks/blob/master/Data%20Analysis/Basic%20Time%20Series.ipynb)

In [5]:
df.set_index(:timestamp, keep: true)
# df.set_index(:distance_cumsum_m, keep: true)
df.head 5

,timestamp,lat,lon,hr,ellapsed_sec,distance_m,speed_m_s,distance_cumsum_m,on_foil
2024-06-15 11:22:15 UTC,2024-06-15 11:22:15 UTC,52.46734491549432,5.33144898712635,97,0.0,1.0e-09,0.0,1.0e-09,0
2024-06-15 11:22:17 UTC,2024-06-15 11:22:17 UTC,52.46731549501419,5.3314343187958,98,2.0,3.4189852354446244,1.71,3.4189852364446245,0
2024-06-15 11:22:20 UTC,2024-06-15 11:22:20 UTC,52.46731985360384,5.331416800618172,98,3.0,1.2818592877714723,0.43,4.700844524216096,0
2024-06-15 11:22:22 UTC,2024-06-15 11:22:22 UTC,52.46731708757579,5.331427529454231,99,2.0,0.7891882503677813,0.39,5.490032774583877,0
2024-06-15 11:22:23 UTC,2024-06-15 11:22:23 UTC,52.46731205843389,5.331429373472929,98,1.0,0.5729970700844821,0.57,6.0630298446683595,0


In [16]:
df.plot(type: :line, x: :distance_cumsum_m, y: :speed_m_s) do |p, d|
  p.x_label 'Distance'
  p.y_label 'Speed'
end

#<Nyaplot::Plot:0x0000004056639278 @properties={:diagrams=>[#<Nyaplot::Diagram:0x0000004055e019c0 @properties={:type=>:line, :options=>{:x=>:distance_cumsum_m, :y=>:speed_m_s}, :data=>"a5b41e81-5ce1-4150-bacc-7c6e9a09a5b3"}, @xrange=[0.0, 23686.57035350084], @yrange=[0.0, 14.75]>], :options=>{:x_label=>"Distance", :y_label=>"Speed", :zoom=>true, :width=>700, :xrange=>[0.0, 23686.57035350084], :yrange=>[0.0, 14.75]}}>

## Analyzing an activity


In [9]:
require 'daru/view'
file = "activities/2024-07-06-wing.gpx"
  fa = FoilActivityAnalyzer.new(file, lift_speed = 15)
  df = fa.data_frame
  dist = df[:distance_cumsum_m]
  speed = df[:speed_m_s] * 3.6
  # line = speed.rolling_mean
  on_foil = df[:on_foil]
  line_graph = Daru::View::Plot.new(
    [
      { type: :area, name: 'On foil', data: dist.to_a.zip((on_foil * speed.max).to_a), showInLegend: false },
      { type: :line, name: 'Speed', data: dist.to_a.zip(speed.to_a) }
      # { type: :line, name: 'HR', data: dist.to_a.zip(df[:hr].to_a) }
      # { type: :line, name: 'Speed AVG', data: dist.to_a.zip(speed.rolling_mean(5).to_a) }
    ],
    title: { text: nil },
    xAxis: [{ title: { text: 'Distance (m)' } }],
    yAxis: [{ title: { text: 'Speed (km/h)' } }],
    tooltip: { enabled: false },
    legend: { enabled: false },
    adapter: :highcharts
  )
line_graph.show_in_iruby

"\n\nvar event = document.createEvent(\"HTMLEvents\");\nevent.initEvent(\"load_highcharts\", false, false);\nwindow.dispatchEvent(event);\nconsole.log(\"Finish loading highchartsjs\");\n"

"              <script type=\"text/javascript\">\n        $(function() {\n\n                var options = { \"title\": { \"text\": null },\"xAxis\": [ { \"title\": { \"text\": \"Distance (m)\" } } ],\"yAxis\": [ { \"title\": { \"text\": \"Speed (km/h)\" } } ],\"tooltip\": { \"enabled\": false },\"legend\": { \"enabled\": false },\"chart\": { \"renderTo\": \"qxlrezsdyth\" },\"series\": [{ \"type\": \"area\",\"name\": \"On foil\",\"data\": [ [ 0.0,0.0 ],[ 1.0581704783339276,0.0 ],[ 4.3969571966098515,0.0 ],[ 13.058801020741278,0.0 ],[ 19.383448050138703,0.0 ],[ 24.71719791624368,0.0 ],[ 27.816964400250917,0.0 ],[ 33.18039089965177,0.0 ],[ 36.7871938954114,0.0 ],[ 40.263081472114024,0.0 ],[ 48.52349977801874,0.0 ],[ 52.59661432178439,0.0 ],[ 65.43699557200249,0.0 ],[ 69.82087917309457,0.0 ],[ 80.11121741946276,0.0 ],[ 85.61826883507155,0.0 ],[ 91.40755949089724,0.0 ],[ 107.28512657435681,0.0 ],[ 117.18495320198339,53.1 ],[ 127.35040682235697,53.1 ],[ 133.3025452515731,53.1 ],[ 138.09687885518952,53.1 ],[ 143.13513391261563,53.1 ],[ 152.95458216175652,53.1 ],[ 163.27210727572768,53.1 ],[ 173.72773771975125,53.1 ],[ 186.33153346835076,53.1 ],[ 196.734228144236,53.1 ],[ 201.4992114049383,53.1 ],[ 206.03797647596662,53.1 ],[ 216.51015399327193,53.1 ],[ 221.94960724027828,53.1 ],[ 231.34037914253315,53.1 ],[ 246.27988894371884,53.1 ],[ 277.5247324956651,53.1 ],[ 282.1863211349932,53.1 ],[ 302.35894122397355,53.1 ],[ 308.6361517868814,53.1 ],[ 313.0645224423872,53.1 ],[ 319.3053707672554,53.1 ],[ 330.88506695440446,53.1 ],[ 335.9228611158278,53.1 ],[ 345.72086339570444,53.1 ],[ 352.10470322531296,53.1 ],[ 363.81024684311694,53.1 ],[ 370.7552485310872,53.1 ],[ 379.43671993647547,53.1 ],[ 383.9179783039084,53.1 ],[ 389.11007513295664,53.1 ],[ 390.890398248277,0.0 ],[ 394.89247550840884,0.0 ],[ 394.9660933012399,0.0 ],[ 396.06148849760496,0.0 ],[ 396.50888123741737,0.0 ],[ 399.03561325612543,0.0 ],[ 399.42079299492036,0.0 ],[ 399.81277414168574,0.0 ],[ 400.0951984148409,0.0 ],[ 401.27140453784176,0.0 ],[ 402.9323766648583,0.0 ],[ 404.6013065876487,0.0 ],[ 407.7474317842159,0.0 ],[ 408.81716148023975,0.0 ],[ 409.3315360274096,0.0 ],[ 411.69628351097305,0.0 ],[ 413.38400103945776,0.0 ],[ 414.0779482413079,0.0 ],[ 415.3154081126925,0.0 ],[ 417.2383231199797,0.0 ],[ 419.79520794966,0.0 ],[ 420.5788606939525,0.0 ],[ 421.04169309023894,0.0 ],[ 421.91539849946287,0.0 ],[ 422.65945858731516,0.0 ],[ 424.86952445179276,0.0 ],[ 425.8202580296356,0.0 ],[ 428.01554256263705,0.0 ],[ 429.4289916428959,0.0 ],[ 430.60679683093144,0.0 ],[ 433.00223747637193,0.0 ],[ 434.30444660186964,0.0 ],[ 436.48082713787255,0.0 ],[ 445.29659580492523,0.0 ],[ 453.3412284287866,0.0 ],[ 457.5099646442263,0.0 ],[ 468.7392275498019,0.0 ],[ 473.8909262570842,0.0 ],[ 477.59467550687566,0.0 ],[ 482.37501802845975,0.0 ],[ 492.1436738635266,0.0 ],[ 502.21478986784484,0.0 ],[ 511.45451751073523,0.0 ],[ 522.9783213319408,53.1 ],[ 528.426145769594,53.1 ],[ 543.0396767191364,53.1 ],[ 547.8166797176732,53.1 ],[ 552.993692482276,53.1 ],[ 558.0240115053475,53.1 ],[ 563.3956529131051,53.1 ],[ 568.6644730206013,53.1 ],[ 575.497524449002,53.1 ],[ 580.456863971335,53.1 ],[ 585.9948732365522,53.1 ],[ 591.4256231915904,53.1 ],[ 606.9699845966435,53.1 ],[ 618.3517719766171,53.1 ],[ 629.7944569082468,53.1 ],[ 635.5206480317008,53.1 ],[ 646.2577334927753,53.1 ],[ 651.4153950302031,53.1 ],[ 657.7065477386654,53.1 ],[ 678.5712114740421,53.1 ],[ 683.7262801501231,53.1 ],[ 693.6066477626612,53.1 ],[ 713.4048271339173,53.1 ],[ 728.9535908418117,53.1 ],[ 738.9470608840559,53.1 ],[ 748.8604967215432,53.1 ],[ 753.5974915153677,53.1 ],[ 759.8110307817444,53.1 ],[ 771.3090633464088,53.1 ],[ 781.6918246084132,53.1 ],[ 787.1946634030671,53.1 ],[ 792.7635156714363,53.1 ],[ 809.6243842377924,53.1 ],[ 815.9085952206556,53.1 ],[ 821.8460102603915,53.1 ],[ 827.8113995884229,53.1 ],[ 845.8479725256267,53.1 ],[ 865.8884316413696,53.1 ],[ 871.0809749381593,53.1 ],[ 876.7676508324812,53.1 ],[ 882.2662355803151,53.1 